# Chatbot Arena: features de forma + LightGBM com troca A/B

Gerado por `scripts/build_kaggle_notebook.py` a partir de [kaggle-llm-preference](https://github.com/pedrohbenetton/kaggle-llm-preference). CV agrupada por prompt: log loss 1,029.

In [ ]:
# ---- lp/data.py ----
"""Leitura e preparação dos dados da Chatbot Arena (LLM Classification Finetuning)."""


import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

TARGETS = ["winner_model_a", "winner_model_b", "winner_tie"]
_LOCAL = (
    str(Path(globals()["__file__"]).resolve().parents[3] / "llm-classification-finetuning"
        / "llm-classification-finetuning")
    if "__file__" in globals()
    else ""
)
_CANDIDATES = (
    os.environ.get("LP_DATA", ""),
    "/kaggle/input/llm-classification-finetuning",
    "/kaggle/input/competitions/llm-classification-finetuning",
    _LOCAL,
)


def data_dir() -> Path:
    for c in _CANDIDATES:
        if c and (Path(c) / "train.csv").exists():
            return Path(c)
    raise FileNotFoundError("train.csv não encontrado; defina LP_DATA")


def join_turns(s: str) -> str:
    """Prompt e respostas vêm como lista JSON (uma entrada por turno, às vezes null)."""
    try:
        turns = json.loads(s)
    except (json.JSONDecodeError, TypeError):
        return str(s)
    if not isinstance(turns, list):
        return str(turns)
    return "\n\n".join("" if t is None else str(t) for t in turns)


def labels(df: pd.DataFrame) -> np.ndarray:
    return df[TARGETS].to_numpy().argmax(1)


def swap(df: pd.DataFrame) -> pd.DataFrame:
    """Troca A e B (respostas, modelos e rótulos). Serve para aumentar dados e medir viés."""
    out = df.copy()
    for a, b in (("response_a", "response_b"), ("model_a", "model_b"),
                 ("winner_model_a", "winner_model_b")):
        if a in out and b in out:
            out[a], out[b] = df[b].to_numpy(), df[a].to_numpy()
    return out


def load(name: str) -> pd.DataFrame:
    return pd.read_csv(data_dir() / f"{name}.csv")

In [ ]:
# ---- lp/features.py ----
"""Features de forma das respostas. `d_*` = A − B (troca de sinal ao trocar A e B),
`s_*` = simétricas. Capturam os vieses conhecidos da Arena: verbosidade, formatação
(markdown, listas, código) e recusas."""


import re

import numpy as np
import pandas as pd


REFUSAL = re.compile(r"\b(I'm sorry|I am sorry|I cannot|I can't|I apologize|As an AI|"
                     r"I'm not able|I am not able|unable to)\b", re.IGNORECASE)
PATTERNS = {
    "bold": r"\*\*", "header": r"(?m)^#{1,6} ", "bullet": r"(?m)^\s*[-*•] ",
    "numbered": r"(?m)^\s*\d+[.)] ", "code": r"```", "newline": r"\n", "question": r"\?",
    "exclaim": r"!", "url": r"https?://",
}


def _one(texts: pd.Series) -> pd.DataFrame:
    f = pd.DataFrame(index=texts.index)
    f["chars"] = texts.str.len()
    f["words"] = texts.str.split().str.len().fillna(0)
    f["log_chars"] = np.log1p(f["chars"])
    for k, p in PATTERNS.items():
        f[k] = texts.str.count(p)
    f["refusal"] = texts.str.count(REFUSAL)
    f["empty"] = (f["chars"] == 0).astype(int)
    return f


def _jaccard(a: str, b: str) -> float:
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / max(1, len(sa | sb))


def features(df: pd.DataFrame) -> pd.DataFrame:
    a = df["response_a"].map(join_turns)
    b = df["response_b"].map(join_turns)
    p = df["prompt"].map(join_turns)
    fa, fb = _one(a), _one(b)
    out = pd.DataFrame(index=df.index)
    for c in fa.columns:
        out[f"d_{c}"] = fa[c] - fb[c]
        out[f"s_{c}"] = fa[c] + fb[c]
    out["d_len_ratio"] = np.log1p(fa["chars"]) - np.log1p(fb["chars"])
    out["s_prompt_chars"] = p.str.len()
    out["s_turns"] = df["prompt"].str.count('","') + 1
    out["s_jaccard"] = [_jaccard(x, y) for x, y in zip(a, b, strict=True)]
    out["s_identical"] = (a == b).astype(int)
    return out.astype(float)

In [ ]:
# ---- lp/model.py ----
"""LightGBM sobre as features de forma, com aumento por troca A↔B: treina com a cópia
trocada e, na previsão, faz a média com a previsão trocada de volta. Isso força a simetria
(P(A vence | A,B) = P(B vence | B,A)) e reduz o viés de posição do modelo."""


import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier



def _swap_features(f: pd.DataFrame) -> pd.DataFrame:
    g = f.copy()
    d = [c for c in f.columns if c.startswith("d_")]
    g[d] = -f[d]
    return g


_SWAP_Y = np.array([1, 0, 2])


class FeatureGBM:
    def __init__(self, **kw):
        params = dict(n_estimators=400, learning_rate=0.03, num_leaves=31, min_child_samples=20,
                      subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
        params.update(kw)
        self.clf = LGBMClassifier(objective="multiclass", **params)

    def fit(self, df: pd.DataFrame, y: np.ndarray):
        f = features(df)
        X = pd.concat([f, _swap_features(f)], ignore_index=True)
        self.clf.fit(X, np.concatenate([y, _SWAP_Y[y]]))
        return self

    def predict_proba(self, df: pd.DataFrame) -> np.ndarray:
        f = features(df)
        p = self.clf.predict_proba(f)
        q = self.clf.predict_proba(_swap_features(f))[:, [1, 0, 2]]
        return (p + q) / 2

In [ ]:
# ---- lp/submission.py ----
"""Arquivo de submissão: id + probabilidades das 3 classes, normalizadas."""


from collections.abc import Sequence
from pathlib import Path

import numpy as np
import pandas as pd



def make_submission(ids: Sequence, probs: np.ndarray) -> pd.DataFrame:
    probs = np.asarray(probs, dtype=float)
    if probs.shape != (len(ids), 3):
        raise ValueError("formato errado")
    if not np.isfinite(probs).all() or (probs < 0).any():
        raise ValueError("probabilidade inválida")
    if len(set(ids)) != len(ids):
        raise ValueError("id repetido")
    probs = probs / probs.sum(1, keepdims=True)
    return pd.DataFrame({"id": list(ids), **dict(zip(TARGETS, probs.T, strict=True))})


def write_submission(s: pd.DataFrame, out_dir: str | Path = ".") -> Path:
    path = Path(out_dir) / "submission.csv"
    s.to_csv(path, index=False)
    return path

In [ ]:
# Treina no train.csv inteiro (com a cópia A<->B) e prevê o teste, com média da previsão trocada.
train, test = load("train"), load("test")
model = FeatureGBM().fit(train, labels(train))
P = model.predict_proba(test)
path = write_submission(make_submission(test["id"].tolist(), P))
print(path, len(test), "linhas")